# 26 · Whirl flutter: what we tried to get the Halo's flutter speed up

A tiltrotor in airplane mode has big, heavy proprotors at the tips of a flexible wing. Above some airspeed, the
wing modes (wing bending, chord bending and torsion) couple with the rotor's flapping and become unstable: **proprotor
whirl flutter**. Certification-style practice requires the aircraft to be flutter-free up to **1.15 × V_D** (FAR/CS 25.629 style).
This notebook uses V_D = 1.25 × V_max, which is an assumption.

This notebook does three things:

1. shows that the sized Halo wing **does not clear** that requirement (262 kt against 302 kt);
2. walks through every lever we tried, one at a time: wing beam, chord and torsion stiffness; torsion stiffness priced
   by the AFDD wing-weight model; nacelle CG and mast length; rotor pitch couplings (delta-3 and pitch-lag);
3. lines them up on one scoreboard: knots gained against kilograms paid.

The model behind it is `aircraft_closure.aeroelasticity.whirl_flutter`. It is a reduced-order proprotor whirl-flutter
model in Johnson's formulation (NASA TN D-7677): a gimballed stiff-inplane rotor with blade lag and coning, delta-3
that changes with collective, pitch-lag coupling, high-inflow blade-element aerodynamics with compressibility, and
follower thrust and torque. It sits on a cantilever wing with a rigid tip nacelle. The model was checked against
NASA's CAMRAD II results for the XV-15 (Acree, Peyran and Johnson, NASA/TP-2004-212262):

- it reproduces the **60 kt drop** from the 23 % thick wing to the 15 % thin wing (57 kt);
- a single **knockdown of 0.84**, fitted on the thick wing, puts the thin wing at 287 kt against CAMRAD's 275 kt.

Every flutter speed below carries that 0.84. The full derivation and the verification are in
`docs/decisions/040-whirl-flutter-model.md`.

**Inputs.** The sized Halo wing, nacelle and rotor come from `output/structure/reference.json`. Write it first with
`python -m examples.halo_structure_reference` (several minutes, run once). Everything in this notebook then runs in
under a minute.

In [ ]:
from tutorial_helpers import *
import numpy as onp
from dataclasses import replace
reference_path = repo_root / "output" / "structure" / "reference.json"
assert reference_path.exists(), "run `python -m examples.halo_structure_reference` from the repo root first"
from aircraft_closure.aeroelasticity.whirl_flutter import whirl_modes, flutter_speed, cantilever_wing_nacelle_airframe
from examples import halo_whirl_flutter as hw
from examples import halo_whirl_flutter_levers as levers
from examples import halo_whirl_flutter_torsion as torsion
ref = levers.reference
kt = hw.knot_m_s
v_max_kt = ref["velocity_max_m_s"] / kt
required_kt = 1.15 * 1.25 * v_max_kt
print(f"V_max {v_max_kt:.0f} kt -> V_D {1.25 * v_max_kt:.0f} kt -> required flutter-free speed 1.15 V_D = {required_kt:.0f} kt")
print(f"wing: span {ref['span_wing_m']:.2f} m, chord {ref['chord_wing_m']:.2f} m, t/c {ref['thickness_to_chord_wing']:.2f}; "
      f"tip mass {ref['mass_tip_kg']:.0f} kg; rotor R {ref['radius_rotor_m']:.2f} m, {ref['count_blades']} blades")
w = ref["wing_afdd"]
print(f"AFDD stiffnesses: EI beam {w['stiffness_beam_Nm2']:.3g}, EI chord {w['stiffness_chord_Nm2']:.3g}, GJ {w['stiffness_torsion_Nm2']:.3g} N m2")

## 1. The starting point

The rotor speed matters, so the check runs at **every airplane-mode rotor speed the mission uses**: reserve loiter,
climb and descent, cruise, climb to ceiling, and maximum speed. For each one, the model sweeps airspeed, finds where
the least-damped mode crosses zero damping, and multiplies that speed by the 0.84 knockdown.

In [ ]:
frame_reference = levers.airframe()
speeds_all = sorted({round(p["speed_rotor_rad_s"], 3) for p in ref["whirl_points"]})
names = {round(p["speed_rotor_rad_s"], 3): p["label"] for p in ref["whirl_points"]}
baseline = levers.flutter_kt(frame_reference, speeds=speeds_all)
for s, (v, label) in zip(speeds_all, baseline):
    print(f"  {s:5.1f} rad/s ({names[s]:<22s}) {v:4.0f} kt  {label}")
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar([f"{s:.1f}" for s in speeds_all], [v for v, _ in baseline], color="C0")
ax.axhline(required_kt, color="C3", ls="--", label=f"1.15 V_D = {required_kt:.0f} kt")
ax.set(xlabel="rotor speed (rad/s)", ylabel="flutter speed (kt, calibrated)", ylim=(200, 330))
ax.legend(); plt.show()
v_baseline = min(v for v, _ in baseline)
check("baseline Halo wing does not clear 1.15 V_D", v_baseline < required_kt)
check("wing beam is the critical mode at every rotor speed", all(label == "wing beam" for _, label in baseline))

Two features of this result shape everything that follows.

- **Wing beam bending is the mode that goes unstable** at every rotor speed. The wing flaps up and down, the nacelle
  pitches with it, and the rotor's hub forces feed energy into that motion.
- **The lowest rotor speed is the worst.** At reserve loiter (23.8 rad/s), a given airspeed means a higher advance
  ratio V/(ΩR), and the destabilizing hub forces grow with it. Maximum speed (39 rad/s) is nearly as bad, so the
  remaining checks run at both ends of the range.

### What "going unstable" looks like

Damping ratio of each mode against airspeed (a V-g plot) at the critical rotor speed. Flutter is where a curve crosses
zero. The airspeed axis carries the 0.84 knockdown so it can be read directly against the requirement.

In [ ]:
speed_critical = speeds_all[0]
rotor_critical = hw.rotor_for(ref["radius_rotor_m"], ref["count_blades"], ref["solidity"], speed_critical)
velocities_kt = onp.arange(150, 420, 5)
curves = {}
for v in velocities_kt:
    m = whirl_modes(frame_reference, rotor_critical, v * kt, speed_critical, hw.density_kg_m3)
    for label, zeta, f in zip(m.labels, m.damping_ratio, m.frequency_rad_s):
        if label.startswith("wing") and f > 0:
            curves.setdefault(label, {})[v] = min(zeta, curves.get(label, {}).get(v, onp.inf))
fig, ax = plt.subplots(figsize=(7, 3.6))
for label, c in curves.items():
    xs = sorted(c); ax.plot(hw.knockdown * onp.array(xs), [c[x] for x in xs], label=label)
ax.axhline(0, color="k", lw=0.8); ax.axvline(required_kt, color="C3", ls="--", label="1.15 V_D")
ax.set(xlabel="airspeed (kt, x 0.84 knockdown)", ylabel="damping ratio", title=f"Halo wing modes at {speed_critical:.1f} rad/s")
ax.legend(); plt.show()

The wing beam curve is the one that crosses zero, at the flutter speed found above and short of the dashed
requirement line. Every lever below is an attempt to push that crossing to the right.

## 2. Attempt 1: stiffen the wing (for free, in the model)

The most obvious lever is to scale each wing stiffness and see what moves. At this point weight is ignored: these
are what-if multipliers on EI beam, EI chord and GJ, one at a time, taking the worse of the two critical rotor
speeds.

In [ ]:
scales = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
length_m = (ref["span_wing_m"] - ref["width_fuselage_m"]) / 2
def frame_scaled(beam=1.0, chord=1.0, torsion_=1.0):
    return cantilever_wing_nacelle_airframe(
        length_m, beam * w["stiffness_beam_Nm2"], chord * w["stiffness_chord_Nm2"], torsion_ * w["stiffness_torsion_Nm2"],
        levers.mass_wing_kg_m, levers.mass_wing_kg_m * (0.25 * levers.chord_m)**2, levers.mass_tip_kg, 0.5,
        (levers.mass_tip_kg * 0.3**2, levers.inertia_cg_kg_m2, levers.inertia_cg_kg_m2), ref["length_mast_m"],
        damping_structural=(0.03, 0.03, 0.03))
stiffness_sweep = {}
for name, key in (("EI beam", "beam"), ("EI chord", "chord"), ("GJ torsion", "torsion_")):
    stiffness_sweep[name] = [min(v for v, _ in levers.flutter_kt(frame_scaled(**{key: s}))) for s in scales]
fig, ax = plt.subplots(figsize=(7, 3.6))
for name, vs in stiffness_sweep.items():
    ax.plot(scales, vs, "o-", label=name)
ax.axhline(required_kt, color="C3", ls="--", label="1.15 V_D")
ax.set(xlabel="stiffness multiplier", ylabel="flutter speed (kt, worst rotor speed)"); ax.legend(); plt.show()
for name, vs in stiffness_sweep.items():
    print(f"  {name:11s} " + "  ".join(f"x{s:<4} {v:4.0f}" for s, v in zip(scales, vs)))
check("chord stiffness does not move the flutter speed (within 3 kt over x0.5-x3)", onp.ptp(stiffness_sweep["EI chord"]) < 3)
check("torsion is a stronger lever than beam at x1.5",
      stiffness_sweep["GJ torsion"][scales.index(1.5)] > stiffness_sweep["EI beam"][scales.index(1.5)])

- **Chord stiffness does nothing.** The chord mode is not involved in this instability.
- **Beam stiffness helps only a little** (x2 gives +10 kt), even though the unstable mode is called wing beam.
- **Torsion stiffness is the real lever.** Wing torsion is what pitches the nacelle and the hub, and hub pitch and yaw
  are what the rotor's whirl forces feed on. Weakening it is punishing (x0.5 loses about 75 kt). Returns diminish
  quickly in the other direction: x3 gains only 30 kt.

So torsion is the stiffness to buy. The next question is what it weighs.

## 3. Attempt 2: buy torsion stiffness, priced by the wing-weight model

The sizing already weighs the wing with the AFDD tiltrotor wing model (NDARC). In that model the torque box is sized
to put the torsion frequency at a target fraction of rotor speed, and the spar caps are sized for beam and chord
bending and for the jump take-off. `examples/halo_whirl_flutter_torsion.py` re-runs that model with a **higher torsion
frequency target**, holding everything else at the Halo reference (and gross weight fixed, so the weight spiral is
not included):

- the realized GJ, EI beam and EI chord go to the flutter model, since a thicker torque box also stiffens bending;
- the change in wing mass is multiplied by the sizing's wing calibration factor (`factor_wing`).

First, check that at its baseline setting the model reproduces the reference wing exactly:

In [ ]:
for k in ("mass_torque_box_kg", "mass_spar_stiffness_kg", "stiffness_torsion_Nm2", "stiffness_beam_Nm2"):
    print(f"  {k:24s} model {float(getattr(torsion.wing_reference, k)):10.4g}   reference {w[k]:10.4g}")
check("AFDD re-run reproduces the reference torque box", abs(float(torsion.wing_reference.mass_torque_box_kg) / w["mass_torque_box_kg"] - 1) < 1e-3)

In [ ]:
frequency_scales = [1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.05, 2.2]
torsion_rows = [torsion.flutter_with_torsion(s) for s in frequency_scales]
print(" torsion freq   GJ      EI beam   flutter   critical at    wing mass")
for s, (v, label, speed, gj, ei, dm) in zip(frequency_scales, torsion_rows):
    print(f"   x{s:<5}     x{gj:5.2f}   x{ei:5.2f}   {v:4.0f} kt   {speed:4.1f} rad/s   {dm:+6.0f} kg")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot([r[5] for r in torsion_rows], [r[0] for r in torsion_rows], "o-", color="C2")
ax.axhline(required_kt, color="C3", ls="--", label="1.15 V_D")
ax.set(xlabel="added wing mass (kg)", ylabel="flutter speed (kt)", title="Buying torsion stiffness"); ax.legend(); plt.show()
v_205, dm_205 = torsion_rows[frequency_scales.index(2.05)][0], torsion_rows[frequency_scales.index(2.05)][5]
check("torsion alone needs about +360 kg of wing to reach 1.15 V_D", abs(v_205 - required_kt) < 4 and 330 < dm_205 < 390)

Two things show up in the table.

- **The first increments are almost free.** The Halo torque box is sitting at its 1 mm composite **minimum gauge**,
  so the wall is already thicker than the torsion-frequency target needs. A small increase in the target just uses up
  that slack.
- **After that, it's expensive.** Clearing 302 kt needs **GJ × 4.2** and about **+360 kg of wing**. That is 5 % of
  take-off mass, and more once the weight spiral is included. Torsion stiffness alone is not a credible fix.

## 4. Attempt 3: pylon geometry

How the nacelle mass sits relative to the wing matters as much as the wing's stiffness. Two geometric levers:

- **Nacelle CG ahead of the tilt spindle.** The reference assumes 0.5 m, which is not yet a Halo layout number. A CG
  forward of the elastic axis couples beam bending into torsion. The sweep holds the nacelle's pitch inertia about its
  own CG fixed.
- **Mast length** (hub ahead of the spindle). A longer arm turns rotor hub forces into larger wing moments.

In [ ]:
cg_values = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75]
mast_values = [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
cg_sweep = onp.array([[v for v, _ in levers.flutter_kt(levers.airframe(x_cg_m=x))] for x in cg_values])
mast_sweep = onp.array([[v for v, _ in levers.flutter_kt(levers.airframe(x_hub_m=x))] for x in mast_values])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for j, s in enumerate(levers.speeds_rad_s):
    axes[0].plot(cg_values, cg_sweep[:, j], "o-", label=f"{s:.1f} rad/s")
    axes[1].plot(mast_values, mast_sweep[:, j], "o-", label=f"{s:.1f} rad/s")
for ax, xl in zip(axes, ("nacelle CG ahead of spindle (m)", "hub ahead of spindle (m)")):
    ax.axhline(required_kt, color="C3", ls="--"); ax.set(xlabel=xl); ax.legend()
axes[0].set_ylabel("flutter speed (kt)"); plt.show()
check("moving the nacelle CG onto the spindle clears 1.15 V_D at maximum-speed rpm but not at reserve loiter",
      cg_sweep[0, 1] > required_kt > cg_sweep[0, 0])
check("a shorter mast helps", mast_sweep[0].min() > mast_sweep[mast_values.index(1.0)].min())

- **Nacelle CG is a strong lever at high rpm.** With the CG on the spindle the maximum-speed rpm clears easily
  (about 320 kt). Reserve loiter only reaches about 277 kt. Every 0.25 m of forward offset costs about 25 kt at high
  rpm, so the real Halo nacelle layout (turbines in the fuselage, an electric drive at each tip) needs to replace the
  0.5 m assumption.
- **Mast length is a modest lever:** 0.6 m instead of 1.0 m gains roughly 12 to 16 kt. It is limited by blade and
  wing clearance in conversion.

## 5. Attempt 4: rotor pitch couplings

The rotor's own kinematics can be changed too.

- **delta-3**, the pitch-flap coupling. When the gimbal flaps, blade pitch changes. The Halo inherits the XV-15's
  -18 deg (effective at the horn level), and it grows with collective in airplane mode.
- **Pitch-lag coupling**, where blade lag changes pitch. The reference is -0.2, as on the XV-15 model.

Neither change costs weight. Both change the rotor's flapping, loads and its other stability modes.

In [ ]:
delta3_values = [-30, -25, -20, -18, -15, -12, -10, -5, 0]
pitch_lag_values = [-0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2]
delta3_sweep = [levers.flutter_kt(pitch_flap_coupling=onp.tan(onp.radians(d))) for d in delta3_values]
pitch_lag_sweep = [levers.flutter_kt(pitch_lag_coupling=k) for k in pitch_lag_values]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for j, s in enumerate(levers.speeds_rad_s):
    axes[0].plot(delta3_values, [r[j][0] for r in delta3_sweep], "o-", label=f"{s:.1f} rad/s")
    axes[1].plot(pitch_lag_values, [r[j][0] for r in pitch_lag_sweep], "o-", label=f"{s:.1f} rad/s")
for ax, xl in zip(axes, ("delta-3 (deg)", "pitch-lag coupling")):
    ax.axhline(required_kt, color="C3", ls="--"); ax.set(xlabel=xl); ax.legend()
axes[0].axvline(-18, color="gray", lw=0.8); axes[1].axvline(-0.2, color="gray", lw=0.8)
axes[0].set_ylabel("flutter speed (kt)"); plt.show()
for d, r in zip(delta3_values, delta3_sweep):
    print(f"  delta-3 {d:+4d} deg: " + "   ".join(f"{v:4.0f} kt ({label})" for v, label in r))
d10 = delta3_sweep[delta3_values.index(-10)]
check("delta-3 of -10 deg clears 1.15 V_D at both critical rotor speeds", min(v for v, _ in d10) > required_kt)
check("at delta-3 = 0 the rotor coning mode, not the wing, becomes critical somewhere",
      any(label != "wing beam" for _, label in delta3_sweep[-1]))

**delta-3 is by far the strongest lever.** Going from -18 deg to -10 deg moves both rotor speeds past 302 kt
(about 325 and 313 kt) without adding any weight. Going further in the wrong direction (-30 deg) costs about 65 kt.
The trend agrees with the published tiltrotor work, where large negative delta-3 is known to erode whirl-flutter
stability. Designers still choose delta-3 mainly to limit flapping, not for flutter.

Before relying on it, note:

- **It is the least validated part of the model.** The 0.84 knockdown was fitted on the XV-15 at -18 deg. Results
  far from that delta-3 are an extrapolation.
- **delta-3 exists for a reason.** Less negative delta-3 means more airplane-mode flapping in gusts and manoeuvres, a
  different gimbal frequency, and possibly ground-resonance and loads issues. At 0 deg the rotor's own coning mode
  takes over as the critical mode. A delta-3 change needs a flapping and loads check.

**Pitch-lag** helps steadily toward positive values (+0.2 gives about +23 kt). The range beyond +0.2 has not been
checked against the collective lag mode. On the XV-15 model, strong pitch-lag destabilized that mode instead of the
wing.

## 6. Scoreboard: knots against kilograms

The worst-case flutter speed over the two critical rotor speeds for each lever at a plausible setting, plus one
combination (delta-3 -10 deg with the nacelle CG moved to 0.25 m) to show the levers add up.

In [ ]:
combo = min(v for v, _ in levers.flutter_kt(levers.airframe(x_cg_m=0.25), pitch_flap_coupling=onp.tan(onp.radians(-10))))
worst = lambda rows: min(v for v, _ in rows)
board = [
    ("reference", v_baseline, 0),
    ("EI chord x2 (what-if)", stiffness_sweep["EI chord"][scales.index(2.0)], None),
    ("EI beam x2 (what-if)", stiffness_sweep["EI beam"][scales.index(2.0)], None),
    ("GJ x2 (AFDD: torsion freq x1.4)", torsion_rows[frequency_scales.index(1.4)][0], torsion_rows[frequency_scales.index(1.4)][5]),
    ("GJ x4.2 (AFDD: torsion freq x2.05)", v_205, dm_205),
    ("mast 0.6 m", mast_sweep[0].min(), None),
    ("nacelle CG on spindle", cg_sweep[0].min(), None),
    ("pitch-lag +0.2", worst(pitch_lag_sweep[-1]), 0),
    ("delta-3 -10 deg", worst(d10), 0),
    ("delta-3 -10 deg + CG 0.25 m", combo, None),
]
fig, ax = plt.subplots(figsize=(8, 4.2))
labels_board = [b[0] for b in board][::-1]; values = [b[1] for b in board][::-1]
colors = ["C2" if v >= required_kt else "C0" for v in values]
ax.barh(labels_board, values, color=colors)
ax.axvline(required_kt, color="C3", ls="--", label=f"1.15 V_D = {required_kt:.0f} kt")
ax.set(xlim=(200, 360), xlabel="flutter speed, worst critical rotor speed (kt)"); ax.legend(loc="lower right"); plt.show()
for name, v, dm in board:
    cost = "free" if dm == 0 else ("not priced" if dm is None else f"+{dm:.0f} kg wing")
    print(f"  {name:36s} {v:4.0f} kt  {v / required_kt - 1:+5.0%}   {cost}")
check("the delta-3 + CG combination clears with more margin than delta-3 alone", combo > worst(d10))

## 7. What we take from it

| Lever | Effect | Cost | Status |
|---|---|---|---|
| EI chord | none | — | dropped |
| EI beam | small | spar caps | dropped |
| GJ torsion | strong, then diminishing | +360 kg to clear on its own | fallback for what remains |
| Nacelle CG toward the spindle | strong at high rpm, weak at loiter rpm | layout | get the real number from the layout |
| Shorter mast | modest | clearance in conversion | secondary |
| Pitch-lag toward + | moderate | rotor stability check | secondary |
| **delta-3 about -10 deg** | **clears on its own** | **flapping and loads** | **proposed, needs checking** |

**Proposed path:** delta-3 of about -10 deg plus the nacelle CG as close to the spindle as the layout allows. Confirm
it on whole-aircraft modes, then use torsion stiffness only for whatever margin is left.

## Limits and next steps

- **Cantilever airframe.** The wing is clamped at the fuselage with a rigid nacelle. On the XV-15 this reads about
  10 % low against NASA's NASTRAN hub modes, so these results are conservative. Next: whole-aircraft symmetric and
  antisymmetric modes, from the CalculiX model of tutorial 25.
- **A single calibration point.** The 0.84 knockdown comes from one aircraft at one delta-3. A second aircraft,
  ideally one with a different delta-3, would show whether the delta-3 trend is trustworthy.
- **No flapping or loads check.** The delta-3 recommendation needs the airplane-mode flapping in gusts and
  manoeuvres, and gimbal-stop margins.
- **Assumed pylon numbers.** Nacelle CG (0.5 m), roll gyration (0.3 m), spindle on the elastic axis, and drive
  inertia (100 kg m²) are placeholders until the layout defines them.
- **Not in the sizing loop yet.** The sizing still uses frequency-placement margins (plan 024), which this wing
  passes, so they miss the shortfall. The next step is a flutter-damping constraint inside the optimizer
  (eigenpair), so that the sizing makes the delta-3, CG and GJ trade itself.

In [ ]:
check_summary()